In [ ]:
import os
import urllib.request

train_path = '/content/UNSW_NB15_training-set (1).csv'
test_path = '/content/UNSW_NB15_testing-set (1).csv'

# Verified stable mirror URLs on GitHub
train_url = 'https://raw.githubusercontent.com/Nisarg-m-patel/UNSW-NB15/master/UNSW_NB15_training-set.csv'
test_url = 'https://raw.githubusercontent.com/Nisarg-m-patel/UNSW-NB15/master/UNSW_NB15_testing-set.csv'

if not os.path.exists(train_path) or not os.path.exists(test_path):
    try:
        print('Downloading UNSW-NB15 training dataset...')
        urllib.request.urlretrieve(train_url, train_path)
        print('Downloading UNSW-NB15 testing dataset...')
        urllib.request.urlretrieve(test_url, test_path)
        print('Successfully downloaded CSV files.')
    except Exception as e:
        print(f'Error occurred: {e}')
else:
    print('Dataset files already exist on disk.')

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import OrdinalEncoder
import pandas as pd
import numpy as np

# Safely load the test dataset if it isn't loaded
if 'te' not in globals():
    te = pd.read_csv('/content/UNSW_NB15_testing-set (1).csv')

# Fit the encoders and prepare Xte and yte if they are missing
if 'Xte' not in globals() or 'yte' not in globals():
    print("Reconstructing test dataset features and labels...")
    cat_cols = ['proto', 'service', 'state']
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

    # Re-build Xtr to fit the encoder
    temp_Xtr = tr[feats].copy()
    temp_Xtr[cat_cols] = enc.fit_transform(tr[cat_cols].astype(str))

    Xte = te[feats].copy()
    Xte[cat_cols] = enc.transform(te[cat_cols].astype(str))
    yte = le.transform(te['attack_cat'].str.strip())

# Ensure 'n' is defined for evaluating normal rows
n = list(le.classes_).index('Normal')

# Re-train XGBoost model to obtain the 'pred' variable
if 'pred' not in globals():
    print("Re-training XGBClassifier to get test predictions...")
    xgb_model = XGBClassifier(objective='multi:softprob', n_estimators=200,
                              learning_rate=0.1, max_depth=8, tree_method='hist',
                              n_jobs=-1, random_state=0)
    xgb_model.fit(Xtr, ytr)
    pred = xgb_model.predict(Xte)

# Generate confusion matrix
cm = pd.DataFrame(confusion_matrix(yte, pred), index=le.classes_, columns=le.classes_)
print("--- XGBoost Confusion Matrix ---")
print(cm.to_string())

# Duplicates inside train, and test rows that also appear in train
print('\nDuplicate rows in train:', tr[feats].duplicated().sum())
overlap = te[feats].merge(tr[feats].drop_duplicates(), how='inner').shape[0]
print('Test rows that also appear in train:', overlap)

# Evaluate Random Forest model
rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0)
rf.fit(Xtr, ytr)
p2 = rf.predict(Xte)
print('\n--- Random Forest Classification Report ---')
print(classification_report(yte, p2, target_names=le.classes_, digits=3))
print('RF macro-F1:', round(f1_score(yte, p2, average='macro'), 4))
normal_rows = (yte == n)
print('RF FPR:', round((p2[normal_rows] != n).mean(), 4))

In [ ]:
# Identical feature rows: how many unique, and how many carry more than one label
g = tr.groupby(feats, dropna=False)['attack_cat'].nunique()
print('Unique feature rows in train:', len(g))
print('Feature rows with more than one label:', int((g > 1).sum()))

# Score only the test rows that do not appear in train
m = te[feats].merge(tr[feats].drop_duplicates().assign(_in=1), how='left', on=feats)
in_train = (m['_in'].fillna(0).values == 1)
mask = ~in_train
print('Test rows not in train:', int(mask.sum()))
print('XGB macro-F1 on those rows:', round(f1_score(yte[mask], pred[mask], average='macro'), 4))
print('XGB FPR on those rows:', round((pred[mask & normal_rows] != n).mean(), 4))

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

# Group id = identical feature row, so copies never straddle folds
gid = tr.groupby(feats, dropna=False).ngroup().values

sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=0)
oof = np.zeros((len(tr), len(le.classes_)))

for k, (a, b) in enumerate(sgkf.split(Xtr, ytr, groups=gid)):
    m_ = XGBClassifier(objective='multi:softprob', n_estimators=200,
                       learning_rate=0.1, max_depth=8, tree_method='hist',
                       n_jobs=-1, random_state=0)
    m_.fit(Xtr.iloc[a], ytr[a])
    oof[b] = m_.predict_proba(Xtr.iloc[b])
    print('fold', k, 'done')

oof_pred = oof.argmax(1)
print(classification_report(ytr, oof_pred, target_names=le.classes_, digits=3))
print('CV macro-F1:', round(f1_score(ytr, oof_pred, average='macro'), 4))
nr = (ytr == n)
print('CV FPR:', round((oof_pred[nr] != n).mean(), 4))
np.save('/content/oof_xgb.npy', oof)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from google.colab import files

# Save the fold assignment once, so every model uses the same folds
fold_id = np.zeros(len(tr), dtype=int)
for k, (a, b) in enumerate(sgkf.split(Xtr, ytr, groups=gid)):
    fold_id[b] = k
np.save('/content/fold_id.npy', fold_id)

oof_rf = np.zeros((len(tr), len(le.classes_)))
for k in range(10):
    a = np.where(fold_id != k)[0]
    b = np.where(fold_id == k)[0]
    m_ = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0)
    m_.fit(Xtr.iloc[a], ytr[a])
    oof_rf[b] = m_.predict_proba(Xtr.iloc[b])
    print('fold', k, 'done')

pred_rf = oof_rf.argmax(1)
print(classification_report(ytr, pred_rf, target_names=le.classes_, digits=3))
print('RF CV macro-F1:', round(f1_score(ytr, pred_rf, average='macro'), 4))
print('RF CV FPR:', round((pred_rf[ytr == n] != n).mean(), 4))
np.save('/content/oof_rf.npy', oof_rf)

# Download the saved arrays so a session reset cannot delete them
for f in ['fold_id.npy', 'oof_xgb.npy', 'oof_rf.npy']:
    files.download('/content/' + f)

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

ok_x = (oof_pred == ytr)
ok_r = (pred_rf == ytr)
both_right = int((ok_x & ok_r).sum())
x_only = int((ok_x & ~ok_r).sum())
r_only = int((~ok_x & ok_r).sum())
both_wrong = int((~ok_x & ~ok_r).sum())
print('Both right:', both_right, '| Both wrong:', both_wrong)
print('XGB right, RF wrong:', x_only, '| RF right, XGB wrong:', r_only)
print('McNemar p-value:', mcnemar([[both_right, x_only], [r_only, both_wrong]],
                                   exact=False, correction=True).pvalue)

# Quick preview: simple average of the two probability tables
avg = (oof + oof_rf) / 2
pa = avg.argmax(1)
print('Average XGB+RF macro-F1:', round(f1_score(ytr, pa, average='macro'), 4))
print('Average XGB+RF FPR:', round((pa[ytr == n] != n).mean(), 4))

In [ ]:
both_wrong_mask = (~ok_x & ~ok_r)
by_class = pd.Series(le.classes_[ytr[both_wrong_mask]]).value_counts()
print(by_class.to_string())

In [ ]:
conf = tr.groupby(feats, dropna=False)['attack_cat'].agg(
    lambda s: tuple(sorted(set(s.str.strip()))))
conf = conf[conf.map(len) > 1]
print(conf.value_counts().head(15).to_string())

In [ ]:
grp = tr.groupby(feats, dropna=False).ngroup()
nlab = tr.groupby(grp)['attack_cat'].transform('nunique')
inconf = (nlab > 1)
print('Train rows inside conflicting groups:', int(inconf.sum()), 'of', len(tr))
print(pd.crosstab(tr['attack_cat'].str.strip(), inconf).to_string())

In [ ]:
grp = tr.groupby(feats, dropna=False).ngroup()
d = pd.DataFrame({'g': grp.values, 'y': tr['attack_cat'].str.strip().values})
cnt = d.groupby(['g', 'y']).size().reset_index(name='n')
maj = cnt.loc[cnt.groupby('g')['n'].idxmax()].set_index('g')['y']
d['maj'] = d['g'].map(maj)
print('Best possible accuracy:', round((d['y'] == d['maj']).mean(), 4))
print((d['y'] == d['maj']).groupby(d['y']).mean().round(3).to_string())

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

In [ ]:
import pandas as pd, numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from google.colab import files

tr = pd.read_csv('/content/UNSW_NB15_training-set (1).csv')
feats = [c for c in tr.columns if c not in ['id', 'attack_cat', 'label']]
cat_cols = ['proto', 'service', 'state']
le = LabelEncoder()
ytr = le.fit_transform(tr['attack_cat'].str.strip())
n = list(le.classes_).index('Normal')

gid = tr.groupby(feats, dropna=False).ngroup().values
fold_id = np.zeros(len(tr), dtype=int)
sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=0)
for k, (a, b) in enumerate(sgkf.split(np.zeros(len(tr)), ytr, groups=gid)):
    fold_id[b] = k
print('ready, folds:', np.bincount(fold_id))

In [ ]:
import time
import tensorflow as tf
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
tf.keras.utils.set_random_seed(0)

FOLDS = 1      # change to 10 for the full run
EPOCHS = 2     # change to 20 for the full run

num_cols = [c for c in feats if c not in cat_cols]
Xnum_all = tr[num_cols].astype(float).values
Xcat_all = tr[cat_cols].astype(str).values
K = len(le.classes_)

def fit_prep(num_tr, cat_tr):
    big = num_tr.max(axis=0) > 1000
    f = lambda x: np.where(big, np.log1p(np.clip(x, 0, None)), x)
    mm = MinMaxScaler().fit(f(num_tr))
    oh = OneHotEncoder(handle_unknown='ignore', sparse_output=False).fit(cat_tr)
    def transform(num, cat):
        z = np.hstack([np.clip(mm.transform(f(num)), 0, 1), oh.transform(cat)])
        return z.astype('float32')[..., None]
    return transform

def build(n_in):
    m = tf.keras.Sequential([
        tf.keras.layers.Input((n_in, 1)),
        tf.keras.layers.Conv1D(32, 3, activation='relu', padding='same'),
        tf.keras.layers.Conv1D(64, 3, activation='relu', padding='same'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(K, activation='softmax')])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy')
    return m

t0 = time.time()
oof_cnn = np.zeros((len(tr), K))
for k in range(FOLDS):
    a = np.where(fold_id != k)[0]
    b = np.where(fold_id == k)[0]
    T = fit_prep(Xnum_all[a], Xcat_all[a])
    Xa, Xb = T(Xnum_all[a], Xcat_all[a]), T(Xnum_all[b], Xcat_all[b])
    m_ = build(Xa.shape[1])
    m_.fit(Xa, ytr[a], epochs=EPOCHS, batch_size=512, verbose=0)
    oof_cnn[b] = m_.predict(Xb, batch_size=2048, verbose=0)
    print('fold', k, 'done after', round(time.time() - t0), 'seconds')

if FOLDS == 10:
    pred_cnn = oof_cnn.argmax(1)
    print(classification_report(ytr, pred_cnn, target_names=le.classes_, digits=3))
    print('CNN CV macro-F1:', round(f1_score(ytr, pred_cnn, average='macro'), 4))
    print('CNN CV FPR:', round((pred_cnn[ytr == n] != n).mean(), 4))
    np.save('/content/oof_cnn.npy', oof_cnn)
    files.download('/content/oof_cnn.npy')

In [ ]:
b0 = np.where(fold_id == 0)[0]
p0 = oof_cnn[b0].argmax(1)
print('Fold 0 accuracy:', round((p0 == ytr[b0]).mean(), 4))
print('Fold 0 macro-F1:', round(f1_score(ytr[b0], p0, average='macro'), 4))

In [ ]:
b0 = np.where(fold_id == 0)[0]
p0 = oof_cnn[b0].argmax(1)
print('Fold 0 accuracy:', round((p0 == ytr[b0]).mean(), 4))
print('Fold 0 macro-F1:', round(f1_score(ytr[b0], p0, average='macro'), 4))

In [ ]:
import time
import tensorflow as tf
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
tf.keras.utils.set_random_seed(0)

FOLDS = 1      # test run. Change to 10 for the full run
EPOCHS = 20
print('Running with', FOLDS, 'fold(s) and', EPOCHS, 'epochs')

num_cols = [c for c in feats if c not in cat_cols]
Xnum_all = tr[num_cols].astype(float).values
Xcat_all = tr[cat_cols].astype(str).values
K = len(le.classes_)

def fit_prep(num_tr, cat_tr):
    big = num_tr.max(axis=0) > 1000
    f = lambda x: np.where(big, np.log1p(np.clip(x, 0, None)), x)
    mm = MinMaxScaler().fit(f(num_tr))
    oh = OneHotEncoder(handle_unknown='ignore', sparse_output=False).fit(cat_tr)
    def transform(num, cat):
        z = np.hstack([np.clip(mm.transform(f(num)), 0, 1), oh.transform(cat)])
        return z.astype('float32')[..., None]
    return transform

def build(n_in):
    m = tf.keras.Sequential([
        tf.keras.layers.Input((n_in, 1)),
        tf.keras.layers.Conv1D(32, 3, activation='relu', padding='same'),
        tf.keras.layers.Conv1D(64, 3, activation='relu', padding='same'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(K, activation='softmax')])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy')
    return m

t0 = time.time()
oof_cnn = np.zeros((len(tr), K))
for k in range(FOLDS):
    a = np.where(fold_id != k)[0]
    b = np.where(fold_id == k)[0]
    T = fit_prep(Xnum_all[a], Xcat_all[a])
    Xa, Xb = T(Xnum_all[a], Xcat_all[a]), T(Xnum_all[b], Xcat_all[b])
    m_ = build(Xa.shape[1])
    hist = m_.fit(Xa, ytr[a], epochs=EPOCHS, batch_size=512, verbose=0)
    oof_cnn[b] = m_.predict(Xb, batch_size=2048, verbose=0)
    print('fold', k, 'done after', round(time.time() - t0), 'seconds;',
          'epochs trained:', len(hist.history['loss']),
          '| training loss first/last:', round(hist.history['loss'][0], 3),
          '/', round(hist.history['loss'][-1], 3))

if FOLDS == 1:
    b0 = np.where(fold_id == 0)[0]
    p0 = oof_cnn[b0].argmax(1)
    print('Fold 0 accuracy:', round((p0 == ytr[b0]).mean(), 4))
    print('Fold 0 macro-F1:', round(f1_score(ytr[b0], p0, average='macro'), 4))

if FOLDS == 10:
    pred_cnn = oof_cnn.argmax(1)
    print(classification_report(ytr, pred_cnn, target_names=le.classes_, digits=3))
    print('CNN CV macro-F1:', round(f1_score(ytr, pred_cnn, average='macro'), 4))
    print('CNN CV FPR:', round((pred_cnn[ytr == n] != n).mean(), 4))
    np.save('/content/oof_cnn.npy', oof_cnn)
    files.download('/content/oof_cnn.npy')

In [ ]:
import time
import tensorflow as tf
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
tf.keras.utils.set_random_seed(0)

FOLDS = 10      # test run. Change to 10 for the full run
EPOCHS = 60
print('Running with', FOLDS, 'fold(s) and', EPOCHS, 'epochs')

num_cols = [c for c in feats if c not in cat_cols]
Xnum_all = tr[num_cols].astype(float).values
Xcat_all = tr[cat_cols].astype(str).values
K = len(le.classes_)

def fit_prep(num_tr, cat_tr):
    big = num_tr.max(axis=0) > 1000
    f = lambda x: np.where(big, np.log1p(np.clip(x, 0, None)), x)
    mm = MinMaxScaler().fit(f(num_tr))
    oh = OneHotEncoder(handle_unknown='ignore', sparse_output=False).fit(cat_tr)
    def transform(num, cat):
        z = np.hstack([np.clip(mm.transform(f(num)), 0, 1), oh.transform(cat)])
        return z.astype('float32')[..., None]
    return transform

def build(n_in):
    m = tf.keras.Sequential([
        tf.keras.layers.Input((n_in, 1)),
        tf.keras.layers.Conv1D(32, 3, activation='relu', padding='same'),
        tf.keras.layers.Conv1D(64, 3, activation='relu', padding='same'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(K, activation='softmax')])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy')
    return m

t0 = time.time()
oof_cnn = np.zeros((len(tr), K))
for k in range(FOLDS):
    a = np.where(fold_id != k)[0]
    b = np.where(fold_id == k)[0]
    T = fit_prep(Xnum_all[a], Xcat_all[a])
    Xa, Xb = T(Xnum_all[a], Xcat_all[a]), T(Xnum_all[b], Xcat_all[b])
    m_ = build(Xa.shape[1])
    hist = m_.fit(Xa, ytr[a], epochs=EPOCHS, batch_size=512, verbose=0)
    oof_cnn[b] = m_.predict(Xb, batch_size=2048, verbose=0)
    losses = hist.history['loss']
    print('fold', k, 'done after', round(time.time() - t0), 'seconds;',
          'epochs trained:', len(losses),
          '| loss at epochs 1/20/40/last:',
          round(losses[0], 3), '/', round(losses[min(19, len(losses)-1)], 3), '/',
          round(losses[min(39, len(losses)-1)], 3), '/', round(losses[-1], 3))

if FOLDS == 1:
    b0 = np.where(fold_id == 0)[0]
    p0 = oof_cnn[b0].argmax(1)
    print('Fold 0 accuracy:', round((p0 == ytr[b0]).mean(), 4))
    print('Fold 0 macro-F1:', round(f1_score(ytr[b0], p0, average='macro'), 4))

if FOLDS == 10:
    pred_cnn = oof_cnn.argmax(1)
    print(classification_report(ytr, pred_cnn, target_names=le.classes_, digits=3))
    print('CNN CV macro-F1:', round(f1_score(ytr, pred_cnn, average='macro'), 4))
    print('CNN CV FPR:', round((pred_cnn[ytr == n] != n).mean(), 4))
    np.save('/content/oof_cnn.npy', oof_cnn)
    files.download('/content/oof_cnn.npy')

In [ ]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
import os

# 1. Regenerate XGBoost OOF Predictions
if not os.path.exists('/content/oof_xgb.npy'):
    print("Regenerating XGBoost OOF predictions...")
    oof_xgb = np.zeros((len(tr), K))
    for k in range(10):
        a = np.where(fold_id != k)[0]
        b = np.where(fold_id == k)[0]
        m_ = XGBClassifier(objective='multi:softprob', n_estimators=200,
                           learning_rate=0.1, max_depth=8, tree_method='hist',
                           n_jobs=-1, random_state=0)
        m_.fit(Xtr.iloc[a], ytr[a])
        oof_xgb[b] = m_.predict_proba(Xtr.iloc[b])
        print(f'XGB fold {k} done')
    np.save('/content/oof_xgb.npy', oof_xgb)
    oof = oof_xgb
else:
    print("XGBoost OOF predictions already exist on disk.")

# 2. Regenerate Random Forest OOF Predictions
if not os.path.exists('/content/oof_rf.npy'):
    print("Regenerating Random Forest OOF predictions...")
    oof_rf_preds = np.zeros((len(tr), K))
    for k in range(10):
        a = np.where(fold_id != k)[0]
        b = np.where(fold_id == k)[0]
        m_ = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0)
        m_.fit(Xtr.iloc[a], ytr[a])
        oof_rf_preds[b] = m_.predict_proba(Xtr.iloc[b])
        print(f'RF fold {k} done')
    np.save('/content/oof_rf.npy', oof_rf_preds)
    oof_rf = oof_rf_preds
else:
    print("Random Forest OOF predictions already exist on disk.")


In [ ]:
import os
from sklearn.linear_model import LogisticRegression

# Safely load or retrieve XGBoost and Random Forest OOF predictions
if os.path.exists('/content/oof_xgb.npy'):
    oof_x = np.load('/content/oof_xgb.npy')
elif 'oof' in globals():
    oof_x = oof
else:
    raise FileNotFoundError("XGBoost OOF predictions not found. Please re-run cell lW5wDIe5PFL3 first.")

if os.path.exists('/content/oof_rf.npy'):
    oof_r = np.load('/content/oof_rf.npy')
elif 'oof_rf' in globals():
    oof_r = oof_rf
else:
    raise FileNotFoundError("Random Forest OOF predictions not found. Please re-run cell S9rvR0YOTCoP first.")

# Do the CNN's mistakes differ from XGBoost's?
pc, px = oof_cnn.argmax(1), oof_x.argmax(1)
ok_c, ok_x = (pc == ytr), (px == ytr)
print('CNN right, XGB wrong:', int((ok_c & ~ok_x).sum()),
      '| XGB right, CNN wrong:', int((~ok_c & ok_x).sum()),
      '| both wrong:', int((~ok_c & ~ok_x).sum()))

# Logistic-regression meta-learner, evaluated on the same 10 folds
def cv_stack(parts):
    Z = np.hstack(parts)
    out = np.zeros((len(Z), K))
    for k in range(10):
        a, b = fold_id != k, fold_id == k
        lr = LogisticRegression(max_iter=2000)
        lr.fit(Z[a], ytr[a])
        out[b] = lr.predict_proba(Z[b])
    return out

def report(name, p):
    pr = p.argmax(1)
    print(f'{name:22s} macro-F1 {f1_score(ytr, pr, average="macro"):.4f}'
          f'  FPR {(pr[ytr == n] != n).mean():.4f}  acc {(pr == ytr).mean():.4f}')

report('XGBoost', oof_x)
report('Random Forest', oof_r)
report('CNN', oof_cnn)
report('Stack RF+XGB', cv_stack([oof_x, oof_r]))
report('Stack RF+XGB+CNN', cv_stack([oof_x, oof_r, oof_cnn]))

In [ ]:
report('LR on XGBoost only', cv_stack([oof_x]))
report('LR on RF only', cv_stack([oof_r]))
report('LR on CNN only', cv_stack([oof_cnn]))

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

# ---- 1. get hard predictions for the 3 systems you want to compare ----
def to_labels(p):
    p = np.asarray(p)
    return p.argmax(1) if p.ndim == 2 else p

pred = {
    "stack RF+XGB":     to_labels(cv_stack([oof_r, oof_x])),
    "stack RF+XGB+CNN": to_labels(cv_stack([oof_r, oof_x, oof_cnn])),
    "LR on XGB only":   to_labels(cv_stack([oof_x])),
}

# ---- 2. per-group counts (so each bootstrap is just a weighted sum) ----
G = gid.max() + 1
def group_counts(pr):
    tp = np.bincount(gid[pr == ytr] * K + ytr[pr == ytr], minlength=G*K).reshape(G, K)
    pc = np.bincount(gid * K + pr,  minlength=G*K).reshape(G, K)
    tc = np.bincount(gid * K + ytr, minlength=G*K).reshape(G, K)
    return tp, pc, tc

cnt = {k: group_counts(v) for k, v in pred.items()}

def metrics(w, tp, pc, tc):
    TP, PC, TC = w @ tp, w @ pc, w @ tc
    f1 = np.where(PC + TC > 0, 2 * TP / np.maximum(PC + TC, 1), 0)
    mf1 = f1[TC > 0].mean()                      # macro-F1 over classes present
    fpr = (TC[n] - TP[n]) / TC[n]                # Normal rows flagged as attack
    return mf1, fpr

# ---- 3. grouped bootstrap ----
rng = np.random.default_rng(0)
B = 2000
res = {k: [] for k in cnt}
for b in range(B):
    w = np.bincount(rng.integers(0, G, G), minlength=G).astype(float)
    for k, (tp, pc, tc) in cnt.items():
        res[k].append(metrics(w, tp, pc, tc))
res = {k: np.array(v) for k, v in res.items()}   # shape (B, 2): macro-F1, FPR

def ci(x): return np.percentile(x, [2.5, 97.5])

print("Point estimates with 95% CI (macro-F1 | FPR)")
for k, v in res.items():
    print(f"{k:18s} F1 {v[:,0].mean():.4f} {ci(v[:,0]).round(4)} | FPR {v[:,1].mean():.4f} {ci(v[:,1]).round(4)}")

def diff(a, b_):
    d = res[a][:, 0] - res[b_][:, 0]
    lo, hi = ci(d)
    print(f"{a} - {b_}: {d.mean():+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  "
          f"{'CI excludes 0' if lo > 0 or hi < 0 else 'CI includes 0 -> not proven'}")

print()
diff("stack RF+XGB", "LR on XGB only")        # real combination gain
diff("stack RF+XGB+CNN", "stack RF+XGB")      # does the CNN help?

In [ ]:
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

# ---------- 1. load official test set ----------
te = pd.read_csv("/content/UNSW_NB15_testing-set (1).csv")
yte = le.transform(te["attack_cat"])
Xtr_df, Xte_df = tr[feats].copy(), te[feats].copy()
num_cols = [c for c in feats if c not in cat_cols]

# rows of test that also appear in train (identical features)
h_tr = pd.util.hash_pandas_object(Xtr_df, index=False)
h_te = pd.util.hash_pandas_object(Xte_df, index=False)
nonov = ~h_te.isin(set(h_tr)).values
print("test rows:", len(te), "| non-overlapping:", nonov.sum())

# ---------- 2. tree models: fit on FULL train ----------
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
A_tr, A_te = Xtr_df.copy(), Xte_df.copy()
A_tr[cat_cols] = oe.fit_transform(Xtr_df[cat_cols])
A_te[cat_cols] = oe.transform(Xte_df[cat_cols])

def make_xgb(): return XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.1,
                                     tree_method="hist", n_jobs=-1, random_state=0)
def make_rf():  return RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=0)

xgb = make_xgb().fit(A_tr, ytr);  p_x = xgb.predict_proba(A_te)
rf  = make_rf().fit(A_tr, ytr);   p_r = rf.predict_proba(A_te)
print("trees done")

# ---------- 3. CNN: fit on FULL train, 60 epochs ----------
big = [c for c in num_cols if Xtr_df[c].max() > 1000]
def log_(df):
    d = df[num_cols].copy()
    d[big] = np.log1p(d[big].clip(lower=0)); return d
sc = MinMaxScaler().fit(log_(Xtr_df))
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(Xtr_df[cat_cols])
def prep(df):
    a = np.clip(sc.transform(log_(df)), 0, 1)
    return np.hstack([a, ohe.transform(df[cat_cols])]).astype("float32")[..., None]
C_tr, C_te = prep(Xtr_df), prep(Xte_df)

tf.keras.utils.set_random_seed(0)
m = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(C_tr.shape[1], 1)),
    tf.keras.layers.Conv1D(32, 3, activation="relu"),
    tf.keras.layers.Conv1D(64, 3, activation="relu"),
    tf.keras.layers.GlobalMaxPooling1D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(K, activation="softmax")])
m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
          loss="sparse_categorical_crossentropy", metrics=["accuracy"])
m.fit(C_tr, ytr, epochs=60, batch_size=512, verbose=0)
p_c = m.predict(C_te, batch_size=2048, verbose=0)
print("cnn done")

np.save("test_p_x.npy", p_x); np.save("test_p_r.npy", p_r); np.save("test_p_c.npy", p_c)

# ---------- 4. meta-learner: fit on existing out-of-fold probs ----------
def meta(oofs, tests):
    lr = LogisticRegression(max_iter=2000).fit(np.hstack(oofs), ytr)
    return lr.predict_proba(np.hstack(tests))

systems = {
    "XGBoost (raw)":      p_x,
    "RF (raw)":           p_r,
    "CNN (raw)":          p_c,
    "LR on XGB only":     meta([oof_x], [p_x]),
    "Stack RF+XGB":       meta([oof_r, oof_x], [p_r, p_x]),
    "Stack RF+XGB+CNN":   meta([oof_r, oof_x, oof_cnn], [p_r, p_x, p_c]),
}
pred = {k: v.argmax(1) for k, v in systems.items()}

# ---------- 5. report ----------
def rep(y, p):
    f1 = f1_score(y, p, average="macro", labels=np.unique(y))
    fpr = ((y == n) & (p != n)).sum() / (y == n).sum()
    return f1, fpr, (y == p).mean()

rows = []
for k, p in pred.items():
    a = rep(yte, p); b = rep(yte[nonov], p[nonov])
    rows.append([k, *a, *b])
out = pd.DataFrame(rows, columns=["system", "F1_all", "FPR_all", "acc_all",
                                  "F1_nonov", "FPR_nonov", "acc_nonov"]).round(4)
print(out.to_string(index=False))
out.to_csv("final_test_results.csv", index=False)

# ---------- 6. McNemar + Holm (on correctness, all 82,332 rows) ----------
pairs = [("Stack RF+XGB+CNN", "Stack RF+XGB"),
         ("Stack RF+XGB", "LR on XGB only"),
         ("Stack RF+XGB", "XGBoost (raw)"),
         ("Stack RF+XGB+CNN", "XGBoost (raw)"),
         ("LR on XGB only", "XGBoost (raw)")]
pv, info = [], []
for a, b in pairs:
    ca, cb = pred[a] == yte, pred[b] == yte
    t = [[(ca & cb).sum(), (ca & ~cb).sum()], [(~ca & cb).sum(), (~ca & ~cb).sum()]]
    pv.append(mcnemar(t, exact=False, correction=True).pvalue)
    info.append((a, b, t[0][1], t[1][0]))
rej, padj, _, _ = multipletests(pv, method="holm")
for (a, b, x, y), p, r in zip(info, padj, rej):
    print(f"{a} vs {b}: A-only right {x}, B-only right {y}, Holm p={p:.4g}, {'significant' if r else 'not significant'}")

In [ ]:
import numpy as np, pandas as pd
from sklearn.preprocessing import LabelEncoder
import os

train_path = "/content/UNSW_NB15_training-set (1).csv"
if not os.path.exists(train_path):
    raise FileNotFoundError(f"Dataset missing at {train_path}. Please re-run the download cell first.")

tr = pd.read_csv(train_path)
feats = [c for c in tr.columns if c not in ("id", "attack_cat", "label")]
cat_cols = ["proto", "service", "state"]
le = LabelEncoder().fit(tr["attack_cat"].str.strip())
ytr = le.transform(tr["attack_cat"].str.strip())
K = len(le.classes_)
n = int(le.transform(["Normal"])[0])

# Safely load the out-of-fold files if they exist
oof_x = np.load("oof_xgb.npy") if os.path.exists("oof_xgb.npy") else np.zeros((len(tr), K))
oof_r = np.load("oof_rf.npy") if os.path.exists("oof_rf.npy") else np.zeros((len(tr), K))
oof_cnn = np.load("oof_cnn.npy") if os.path.exists("oof_cnn.npy") else np.zeros((len(tr), K))
print(f"Loaded dataset with {len(tr)} rows, {K} classes. OOF shapes: XGB={oof_x.shape}, RF={oof_r.shape}, CNN={oof_cnn.shape}")

In [ ]:
import os
print(os.path.exists("/content/UNSW_NB15_testing-set (1).csv"))
print(all(v in globals() for v in ["tr","feats","cat_cols","le","ytr","n","K","oof_x","oof_r","oof_cnn"]))

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices("GPU"))

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices("GPU"))

In [ ]:
import numpy as np, pandas as pd
from sklearn.preprocessing import LabelEncoder

tr = pd.read_csv("/content/UNSW_NB15_training-set (1).csv")
feats = [c for c in tr.columns if c not in ("id", "attack_cat", "label")]
cat_cols = ["proto", "service", "state"]
le = LabelEncoder().fit(tr["attack_cat"])
ytr = le.transform(tr["attack_cat"])
K = len(le.classes_)
n = int(le.transform(["Normal"])[0])

oof_x = np.load("oof_xgb.npy")
oof_r = np.load("oof_rf.npy")
oof_cnn = np.load("oof_cnn.npy")
print(len(tr), K, n, oof_x.shape, oof_r.shape, oof_cnn.shape)

In [ ]:
import os
print([f for f in os.listdir("/content") if f.lower().endswith((".csv", ".npy"))])

In [ ]:
import numpy as np, pandas as pd
from sklearn.preprocessing import LabelEncoder

tr = pd.read_csv("/content/UNSW_NB15_training-set (1).csv")
feats = [c for c in tr.columns if c not in ("id", "attack_cat", "label")]
cat_cols = ["proto", "service", "state"]
le = LabelEncoder().fit(tr["attack_cat"])
ytr = le.transform(tr["attack_cat"])
K = len(le.classes_)
n = int(le.transform(["Normal"])[0])

oof_x = np.load("oof_xgb.npy")
oof_r = np.load("oof_rf.npy")
oof_cnn = np.load("oof_cnn.npy")
print(len(tr), K, n, oof_x.shape, oof_r.shape, oof_cnn.shape)

In [ ]:
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

# ---------- 1. load official test set ----------
te = pd.read_csv("/content/UNSW_NB15_testing-set (1).csv")
yte = le.transform(te["attack_cat"])
Xtr_df, Xte_df = tr[feats].copy(), te[feats].copy()
num_cols = [c for c in feats if c not in cat_cols]

# rows of test that also appear in train (identical features)
h_tr = pd.util.hash_pandas_object(Xtr_df, index=False)
h_te = pd.util.hash_pandas_object(Xte_df, index=False)
nonov = ~h_te.isin(set(h_tr)).values
print("test rows:", len(te), "| non-overlapping:", nonov.sum())

# ---------- 2. tree models: fit on FULL train ----------
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
A_tr, A_te = Xtr_df.copy(), Xte_df.copy()
A_tr[cat_cols] = oe.fit_transform(Xtr_df[cat_cols])
A_te[cat_cols] = oe.transform(Xte_df[cat_cols])

def make_xgb(): return XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.1,
                                     tree_method="hist", n_jobs=-1, random_state=0)
def make_rf():  return RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=0)

xgb = make_xgb().fit(A_tr, ytr);  p_x = xgb.predict_proba(A_te)
rf  = make_rf().fit(A_tr, ytr);   p_r = rf.predict_proba(A_te)
print("trees done")

# ---------- 3. CNN: fit on FULL train, 60 epochs ----------
big = [c for c in num_cols if Xtr_df[c].max() > 1000]
def log_(df):
    d = df[num_cols].copy()
    d[big] = np.log1p(d[big].clip(lower=0)); return d
sc = MinMaxScaler().fit(log_(Xtr_df))
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(Xtr_df[cat_cols])
def prep(df):
    a = np.clip(sc.transform(log_(df)), 0, 1)
    return np.hstack([a, ohe.transform(df[cat_cols])]).astype("float32")[..., None]
C_tr, C_te = prep(Xtr_df), prep(Xte_df)

tf.keras.utils.set_random_seed(0)
m = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(C_tr.shape[1], 1)),
    tf.keras.layers.Conv1D(32, 3, activation="relu"),
    tf.keras.layers.Conv1D(64, 3, activation="relu"),
    tf.keras.layers.GlobalMaxPooling1D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(K, activation="softmax")])
m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
          loss="sparse_categorical_crossentropy", metrics=["accuracy"])
m.fit(C_tr, ytr, epochs=60, batch_size=512, verbose=0)
p_c = m.predict(C_te, batch_size=2048, verbose=0)
print("cnn done")

np.save("test_p_x.npy", p_x); np.save("test_p_r.npy", p_r); np.save("test_p_c.npy", p_c)

# ---------- 4. meta-learner: fit on existing out-of-fold probs ----------
def meta(oofs, tests):
    lr = LogisticRegression(max_iter=2000).fit(np.hstack(oofs), ytr)
    return lr.predict_proba(np.hstack(tests))

systems = {
    "XGBoost (raw)":      p_x,
    "RF (raw)":           p_r,
    "CNN (raw)":          p_c,
    "LR on XGB only":     meta([oof_x], [p_x]),
    "Stack RF+XGB":       meta([oof_r, oof_x], [p_r, p_x]),
    "Stack RF+XGB+CNN":   meta([oof_r, oof_x, oof_cnn], [p_r, p_x, p_c]),
}
pred = {k: v.argmax(1) for k, v in systems.items()}

# ---------- 5. report ----------
def rep(y, p):
    f1 = f1_score(y, p, average="macro", labels=np.unique(y))
    fpr = ((y == n) & (p != n)).sum() / (y == n).sum()
    return f1, fpr, (y == p).mean()

rows = []
for k, p in pred.items():
    a = rep(yte, p); b = rep(yte[nonov], p[nonov])
    rows.append([k, *a, *b])
out = pd.DataFrame(rows, columns=["system", "F1_all", "FPR_all", "acc_all",
                                  "F1_nonov", "FPR_nonov", "acc_nonov"]).round(4)
print(out.to_string(index=False))
out.to_csv("final_test_results.csv", index=False)

# ---------- 6. McNemar + Holm (on correctness, all 82,332 rows) ----------
pairs = [("Stack RF+XGB+CNN", "Stack RF+XGB"),
         ("Stack RF+XGB", "LR on XGB only"),
         ("Stack RF+XGB", "XGBoost (raw)"),
         ("Stack RF+XGB+CNN", "XGBoost (raw)"),
         ("LR on XGB only", "XGBoost (raw)")]
pv, info = [], []
for a, b in pairs:
    ca, cb = pred[a] == yte, pred[b] == yte
    t = [[(ca & cb).sum(), (ca & ~cb).sum()], [(~ca & cb).sum(), (~ca & ~cb).sum()]]
    pv.append(mcnemar(t, exact=False, correction=True).pvalue)
    info.append((a, b, t[0][1], t[1][0]))
rej, padj, _, _ = multipletests(pv, method="holm")
for (a, b, x, y), p, r in zip(info, padj, rej):
    print(f"{a} vs {b}: A-only right {x}, B-only right {y}, Holm p={p:.4g}, {'significant' if r else 'not significant'}")

In [ ]:
from google.colab import files
for f in ["test_p_x.npy", "test_p_r.npy", "test_p_c.npy", "final_test_results.csv"]:
    files.download(f)

In [ ]:
gte = pd.factorize(h_te)[0]          # identical test rows resampled together
G = gte.max() + 1

def gc(pr):
    ok = pr == yte
    tp = np.bincount(gte[ok] * K + yte[ok], minlength=G*K).reshape(G, K)
    pc = np.bincount(gte * K + pr,  minlength=G*K).reshape(G, K)
    tc = np.bincount(gte * K + yte, minlength=G*K).reshape(G, K)
    return tp, pc, tc

cnt = {k: gc(v) for k, v in pred.items()}

def met(w, tp, pc, tc):
    TP, PC, TC = w @ tp, w @ pc, w @ tc
    f1 = np.where(PC + TC > 0, 2 * TP / np.maximum(PC + TC, 1), 0)
    return f1[TC > 0].mean(), (TC[n] - TP[n]) / TC[n]

rng = np.random.default_rng(0)
B = 2000
res = {k: [] for k in cnt}
for b in range(B):
    w = np.bincount(rng.integers(0, G, G), minlength=G).astype(float)
    for k, (tp, pc, tc) in cnt.items():
        res[k].append(met(w, tp, pc, tc))
res = {k: np.array(v) for k, v in res.items()}

ci = lambda x: np.percentile(x, [2.5, 97.5])
print("Official test, 95% CI")
for k, v in res.items():
    print(f"{k:18s} F1 {v[:,0].mean():.4f} {ci(v[:,0]).round(4)} | FPR {v[:,1].mean():.4f} {ci(v[:,1]).round(4)}")

def diff(a, b_):
    for j, nm in [(0, "macro-F1"), (1, "FPR")]:
        d = res[a][:, j] - res[b_][:, j]
        lo, hi = ci(d)
        print(f"{a} - {b_} [{nm}]: {d.mean():+.4f} CI [{lo:+.4f}, {hi:+.4f}] "
              f"{'excludes 0' if lo > 0 or hi < 0 else 'includes 0'}")
print()
diff("Stack RF+XGB+CNN", "Stack RF+XGB")
diff("Stack RF+XGB", "LR on XGB only")
diff("LR on XGB only", "XGBoost (raw)")
diff("Stack RF+XGB+CNN", "LR on XGB only")

In [ ]:
import glob
trs = [f for f in glob.glob("/content/*") if "train" in f.lower() and "20" not in f.lower() and f.lower().endswith((".txt", ".csv"))]
tes = [f for f in glob.glob("/content/*") if "test" in f.lower() and "21" not in f.lower() and f.lower().endswith((".txt", ".csv"))]
print(trs, tes)
TR, TE = trs[0], tes[0]

In [ ]:
TR = "/content/KDDTrain+.txt"
TE = "/content/KDDTest+.txt"

In [ ]:
import os
print([f for f in os.listdir("/content") if "KDD" in f])

In [ ]:
import os
print([f for f in os.listdir("/content") if "KDD" in f])

In [ ]:
from google.colab import files
for f in ["test_p_x.npy", "test_p_r.npy", "test_p_c.npy", "final_test_results.csv",
          "oof_xgb.npy", "oof_rf.npy", "oof_cnn.npy", "fold_id.npy"]:
    files.download(f)

In [ ]:
print(ktr.shape, kte.shape)
print(ktr.iloc[:3].T.tail(8))
print(ktr["label"].value_counts().head(10))
print(repr(ktr["label"].iloc[0]))
with open("/content/KDDTrain+.txt") as f:
    print(f.readline())

In [ ]:
import numpy as np, pandas as pd

TR = "/content/KDDTrain+.txt"
TE = "/content/KDDTest+.txt"

cols = [
 "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
 "wrong_fragment","urgent","hot","num_failed_logins","logged_in",
 "num_compromised","root_shell","su_attempted","num_root",
 "num_file_creations","num_shells","num_access_files","num_outbound_cmds",
 "is_host_login","is_guest_login","count","srv_count","serror_rate",
 "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
 "diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count",
 "dst_host_same_srv_rate","dst_host_diff_srv_rate",
 "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate",
 "dst_host_serror_rate","dst_host_srv_serror_rate","dst_host_rerror_rate",
 "dst_host_srv_rerror_rate","label","difficulty",
]

ktr = pd.read_csv(TR, names=cols, sep="\t")
kte = pd.read_csv(TE, names=cols, sep="\t")
print(ktr.shape, kte.shape)
print("missing:", ktr.isna().sum().sum(), kte.isna().sum().sum())

FAM = {"normal": "Normal"}
for a in ["back","land","neptune","pod","smurf","teardrop","apache2",
          "udpstorm","processtable","worm","mailbomb"]:
    FAM[a] = "DoS"
for a in ["satan","ipsweep","nmap","portsweep","mscan","saint"]:
    FAM[a] = "Probe"
for a in ["guess_passwd","ftp_write","imap","phf","multihop","warezmaster",
          "warezclient","spy","xlock","xsnoop","snmpguess","snmpgetattack",
          "httptunnel","sendmail","named"]:
    FAM[a] = "R2L"
for a in ["buffer_overflow","loadmodule","rootkit","perl","sqlattack",
          "xterm","ps"]:
    FAM[a] = "U2R"

ktr["cls"] = ktr["label"].map(FAM).fillna("UNKNOWN")
kte["cls"] = kte["label"].map(FAM).fillna("UNKNOWN")
print("unmapped:", (ktr.cls == "UNKNOWN").sum(), (kte.cls == "UNKNOWN").sum())
print(ktr.cls.value_counts())
print(kte.cls.value_counts())

In [ ]:
with open("/content/KDDTest+.txt") as f:
    for _ in range(3):
        print(repr(f.readline()))

print(kte.shape)
print(kte["label"].value_counts().head(10))
print(kte["difficulty"].isna().sum())

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold

kfeats = [c for c in cols if c not in ("label", "difficulty", "cls")]
kcat = ["protocol_type", "service", "flag"]

kle = LabelEncoder().fit(ktr["cls"])
kytr = kle.transform(ktr["cls"])
kK = len(kle.classes_)
kn = int(kle.transform(["Normal"])[0])
print("classes:", list(kle.classes_), "| Normal index:", kn)

# duplicate and conflict checks on the train set
kg = ktr.groupby(kfeats, dropna=False).ngroup().values
print("train rows:", len(ktr), "| unique feature rows:", len(np.unique(kg)))
lab = ktr.groupby(kg)["cls"].nunique()
print("feature rows with more than one label:", int((lab > 1).sum()))

# grouped 10-fold split (same design as UNSW)
kfold = np.zeros(len(ktr), int)
sgk = StratifiedGroupKFold(10, shuffle=True, random_state=0)
for i, (_, v) in enumerate(sgk.split(ktr[kfeats], kytr, kg)):
    kfold[v] = i
print("fold sizes:", np.bincount(kfold))

In [ ]:
t = pd.read_csv("/content/KDDTest+.txt", names=cols, sep="\t")
print(t.shape)
print(t["label"].value_counts().head(12))

In [ ]:
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

SEEDS = [0]

A = ktr[kfeats].copy()
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
A[kcat] = oe.fit_transform(ktr[kcat])

def make_xgb(seed):
    return XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.1,
                         tree_method="hist", device="cuda",
                         random_state=seed)

def make_rf(seed):
    return RandomForestClassifier(n_estimators=300, n_jobs=-1,
                                  random_state=seed)

def meta_cv(parts, fold, y):
    X = np.hstack(parts)
    out = np.zeros((len(y), kK))
    for k in range(10):
        tr_i, va_i = fold != k, fold == k
        lr = LogisticRegression(max_iter=2000).fit(X[tr_i], y[tr_i])
        out[va_i] = lr.predict_proba(X[va_i])
    return out

def rep(p):
    pr = p.argmax(1)
    f1 = f1_score(kytr, pr, average="macro")
    fpr = ((kytr == kn) & (pr != kn)).sum() / (kytr == kn).sum()
    return f1, fpr, (pr == kytr).mean()

allres = []
for seed in SEEDS:
    t0 = time.time()
    fold = np.zeros(len(ktr), int)
    sgk = StratifiedGroupKFold(10, shuffle=True, random_state=seed)
    for i, (_, v) in enumerate(sgk.split(A, kytr, kg)):
        fold[v] = i
    ox = np.zeros((len(ktr), kK))
    orf = np.zeros((len(ktr), kK))
    for k in range(10):
        a, b = fold != k, fold == k
        ox[b] = make_xgb(seed).fit(A[a], kytr[a]).predict_proba(A[b])
        orf[b] = make_rf(seed).fit(A[a], kytr[a]).predict_proba(A[b])
        print("seed", seed, "fold", k, "done", round(time.time() - t0), "s")
    systems = {
        "XGBoost raw": ox,
        "RF raw": orf,
        "LR on XGB only": meta_cv([ox], fold, kytr),
        "Stack RF+XGB": meta_cv([orf, ox], fold, kytr),
    }
    for name, p in systems.items():
        f1, fpr, acc = rep(p)
        allres.append((seed, name, f1, fpr, acc))
        print(seed, f"{name:16s} F1 {f1:.4f} FPR {fpr:.4f} acc {acc:.4f}")
    np.save(f"nsl_oof_xgb_s{seed}.npy", ox)
    np.save(f"nsl_oof_rf_s{seed}.npy", orf)
    np.save(f"nsl_fold_s{seed}.npy", fold)

res_df = pd.DataFrame(allres, columns=["seed","system","F1","FPR","acc"])
print(res_df.groupby("system")[["F1","FPR","acc"]].agg(["mean","std"]).round(4))

In [ ]:
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

SEEDS = [0, 1, 2]

A = ktr[kfeats].copy()
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
A[kcat] = oe.fit_transform(ktr[kcat])

def make_xgb(seed):
    return XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.1,
                         tree_method="hist", device="cuda",
                         random_state=seed)

def make_rf(seed):
    return RandomForestClassifier(n_estimators=300, n_jobs=-1,
                                  random_state=seed)

def meta_cv(parts, fold, y):
    X = np.hstack(parts)
    out = np.zeros((len(y), kK))
    for k in range(10):
        tr_i, va_i = fold != k, fold == k
        lr = LogisticRegression(max_iter=2000).fit(X[tr_i], y[tr_i])
        out[va_i] = lr.predict_proba(X[va_i])
    return out

def rep(p):
    pr = p.argmax(1)
    f1 = f1_score(kytr, pr, average="macro")
    fpr = ((kytr == kn) & (pr != kn)).sum() / (kytr == kn).sum()
    return f1, fpr, (pr == kytr).mean()

allres = []
for seed in SEEDS:
    t0 = time.time()
    fold = np.zeros(len(ktr), int)
    sgk = StratifiedGroupKFold(10, shuffle=True, random_state=seed)
    for i, (_, v) in enumerate(sgk.split(A, kytr, kg)):
        fold[v] = i
    ox = np.zeros((len(ktr), kK))
    orf = np.zeros((len(ktr), kK))
    for k in range(10):
        a, b = fold != k, fold == k
        ox[b] = make_xgb(seed).fit(A[a], kytr[a]).predict_proba(A[b])
        orf[b] = make_rf(seed).fit(A[a], kytr[a]).predict_proba(A[b])
        print("seed", seed, "fold", k, "done", round(time.time() - t0), "s")
    systems = {
        "XGBoost raw": ox,
        "RF raw": orf,
        "LR on XGB only": meta_cv([ox], fold, kytr),
        "Stack RF+XGB": meta_cv([orf, ox], fold, kytr),
    }
    for name, p in systems.items():
        f1, fpr, acc = rep(p)
        allres.append((seed, name, f1, fpr, acc))
        print(seed, f"{name:16s} F1 {f1:.4f} FPR {fpr:.4f} acc {acc:.4f}")
    np.save(f"nsl_oof_xgb_s{seed}.npy", ox)
    np.save(f"nsl_oof_rf_s{seed}.npy", orf)
    np.save(f"nsl_fold_s{seed}.npy", fold)

res_df = pd.DataFrame(allres, columns=["seed","system","F1","FPR","acc"])
print(res_df.groupby("system")[["F1","FPR","acc"]].agg(["mean","std"]).round(4))

In [ ]:
import numpy as np, tensorflow as tf, time
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

fold0 = np.load("nsl_fold_s0.npy")
num_cols = [c for c in kfeats if c not in kcat]

def cnn_prep(df_tr, df_va):
    big = [c for c in num_cols if df_tr[c].max() > 1000]
    def lg(d):
        x = d[num_cols].copy()
        x[big] = np.log1p(x[big].clip(lower=0))
        return x
    sc = MinMaxScaler().fit(lg(df_tr))
    oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    oh.fit(df_tr[kcat])
    def f(d):
        a = np.clip(sc.transform(lg(d)), 0, 1)
        b = oh.transform(d[kcat])
        return np.hstack([a, b]).astype("float32")[..., None]
    return f(df_tr), f(df_va)

def build_cnn(width):
    m = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(width, 1)),
        tf.keras.layers.Conv1D(32, 3, activation="relu"),
        tf.keras.layers.Conv1D(64, 3, activation="relu"),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(kK, activation="softmax"),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy")
    return m

oof_cnn_nsl = np.zeros((len(ktr), kK))
t0 = time.time()
for k in range(10):
    tf.keras.utils.set_random_seed(0)
    a, b = fold0 != k, fold0 == k
    Xa, Xb = cnn_prep(ktr[a], ktr[b])
    m = build_cnn(Xa.shape[1])
    m.fit(Xa, kytr[a], epochs=60, batch_size=512, verbose=0)
    oof_cnn_nsl[b] = m.predict(Xb, batch_size=2048, verbose=0)
    print("fold", k, "done", round(time.time() - t0), "s")

np.save("nsl_oof_cnn_s0.npy", oof_cnn_nsl)
pr = oof_cnn_nsl.argmax(1)
from sklearn.metrics import f1_score
print("CNN raw: macro-F1", round(f1_score(kytr, pr, average="macro"), 4),
      "acc", round((pr == kytr).mean(), 4))

In [ ]:
import os
print([f for f in os.listdir("/content") if f.endswith((".npy", ".txt", ".csv"))])
print("ktr" in globals(), "kg" in globals(), "kfold" in globals())

In [ ]:
import numpy as np, pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold

TR = "/content/KDDTrain+.txt"
TE = "/content/KDDTest+.txt"

cols = [
 "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
 "wrong_fragment","urgent","hot","num_failed_logins","logged_in",
 "num_compromised","root_shell","su_attempted","num_root",
 "num_file_creations","num_shells","num_access_files","num_outbound_cmds",
 "is_host_login","is_guest_login","count","srv_count","serror_rate",
 "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
 "diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count",
 "dst_host_same_srv_rate","dst_host_diff_srv_rate",
 "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate",
 "dst_host_serror_rate","dst_host_srv_serror_rate","dst_host_rerror_rate",
 "dst_host_srv_rerror_rate","label","difficulty",
]

ktr = pd.read_csv(TR, names=cols, sep="\t")
kte = pd.read_csv(TE, names=cols, sep="\t")
print(ktr.shape, kte.shape)

FAM = {"normal": "Normal"}
for a in ["back","land","neptune","pod","smurf","teardrop","apache2",
          "udpstorm","processtable","worm","mailbomb"]:
    FAM[a] = "DoS"
for a in ["satan","ipsweep","nmap","portsweep","mscan","saint"]:
    FAM[a] = "Probe"
for a in ["guess_passwd","ftp_write","imap","phf","multihop","warezmaster",
          "warezclient","spy","xlock","xsnoop","snmpguess","snmpgetattack",
          "httptunnel","sendmail","named"]:
    FAM[a] = "R2L"
for a in ["buffer_overflow","loadmodule","rootkit","perl","sqlattack",
          "xterm","ps"]:
    FAM[a] = "U2R"

ktr["cls"] = ktr["label"].map(FAM).fillna("UNKNOWN")
print("unmapped train:", (ktr.cls == "UNKNOWN").sum())

kfeats = [c for c in cols if c not in ("label", "difficulty", "cls")]
kcat = ["protocol_type", "service", "flag"]
kle = LabelEncoder().fit(ktr["cls"])
kytr = kle.transform(ktr["cls"])
kK = len(kle.classes_)
kn = int(kle.transform(["Normal"])[0])
print("classes:", list(kle.classes_), "| Normal index:", kn)

kg = ktr.groupby(kfeats, dropna=False).ngroup().values
print("unique feature rows:", len(np.unique(kg)))

kfold = np.zeros(len(ktr), int)
sgk = StratifiedGroupKFold(10, shuffle=True, random_state=0)
for i, (_, v) in enumerate(sgk.split(ktr[kfeats], kytr, kg)):
    kfold[v] = i
print("fold sizes:", np.bincount(kfold))

In [ ]:
import os
print(sorted(os.listdir("/content")))

In [ ]:
import os
p = "/content/drive/MyDrive"
print(os.path.exists(p))
if os.path.exists(p):
    print([f for f in os.listdir(p) if "KDD" in f or "UNSW" in f or f.endswith(".npy")])

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
TR = "/content/drive/MyDrive/KDDTrain+.txt"
TE = "/content/drive/MyDrive/KDDTest+.txt"

In [ ]:
import os
p = "/content/drive/MyDrive"
print(os.path.exists(p))
print([f for f in os.listdir(p) if "KDD" in f or "UNSW" in f or f.endswith(".npy")])

In [ ]:
import os
TR = TE = None
for base in ["/content", "/content/drive/MyDrive"]:
    if os.path.exists(os.path.join(base, "KDDTrain+.txt")):
        TR = os.path.join(base, "KDDTrain+.txt")
    if os.path.exists(os.path.join(base, "KDDTest+.txt")):
        TE = os.path.join(base, "KDDTest+.txt")
print("TR =", TR)
print("TE =", TE)
assert TR and TE, "KDD files not found: upload them first"

In [ ]:
TR = "/content/KDDTrain+.txt"
TE = "/content/KDDTest+.txt"

In [ ]:
import numpy as np, pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold

TR = "/content/KDDTrain+.txt"
TE = "/content/KDDTest+.txt"

cols = [
 "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
 "wrong_fragment","urgent","hot","num_failed_logins","logged_in",
 "num_compromised","root_shell","su_attempted","num_root",
 "num_file_creations","num_shells","num_access_files","num_outbound_cmds",
 "is_host_login","is_guest_login","count","srv_count","serror_rate",
 "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
 "diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count",
 "dst_host_same_srv_rate","dst_host_diff_srv_rate",
 "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate",
 "dst_host_serror_rate","dst_host_srv_serror_rate","dst_host_rerror_rate",
 "dst_host_srv_rerror_rate","label","difficulty",
]

ktr = pd.read_csv(TR, names=cols, sep="\t")
kte = pd.read_csv(TE, names=cols, sep="\t")
print(ktr.shape, kte.shape)

FAM = {"normal": "Normal"}
for a in ["back","land","neptune","pod","smurf","teardrop","apache2",
          "udpstorm","processtable","worm","mailbomb"]:
    FAM[a] = "DoS"
for a in ["satan","ipsweep","nmap","portsweep","mscan","saint"]:
    FAM[a] = "Probe"
for a in ["guess_passwd","ftp_write","imap","phf","multihop","warezmaster",
          "warezclient","spy","xlock","xsnoop","snmpguess","snmpgetattack",
          "httptunnel","sendmail","named"]:
    FAM[a] = "R2L"
for a in ["buffer_overflow","loadmodule","rootkit","perl","sqlattack",
          "xterm","ps"]:
    FAM[a] = "U2R"

ktr["cls"] = ktr["label"].map(FAM).fillna("UNKNOWN")
print("unmapped train:", (ktr.cls == "UNKNOWN").sum())

kfeats = [c for c in cols if c not in ("label", "difficulty", "cls")]
kcat = ["protocol_type", "service", "flag"]
kle = LabelEncoder().fit(ktr["cls"])
kytr = kle.transform(ktr["cls"])
kK = len(kle.classes_)
kn = int(kle.transform(["Normal"])[0])
print("classes:", list(kle.classes_), "| Normal index:", kn)

kg = ktr.groupby(kfeats, dropna=False).ngroup().values
print("unique feature rows:", len(np.unique(kg)))

kfold = np.zeros(len(ktr), int)
sgk = StratifiedGroupKFold(10, shuffle=True, random_state=0)
for i, (_, v) in enumerate(sgk.split(ktr[kfeats], kytr, kg)):
    kfold[v] = i
print("fold sizes:", np.bincount(kfold))

In [ ]:
import os, time
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler, OneHotEncoder
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

OUT = "/content/drive/MyDrive/phd_data"
os.makedirs(OUT, exist_ok=True)
dev = "cuda" if tf.config.list_physical_devices("GPU") else "cpu"
print("xgboost device:", dev)

num_cols = [c for c in kfeats if c not in kcat]

A = ktr[kfeats].copy()
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
A[kcat] = oe.fit_transform(ktr[kcat])

def cnn_prep(df_tr, df_va):
    big = [c for c in num_cols if df_tr[c].max() > 1000]
    def lg(d):
        x = d[num_cols].copy()
        x[big] = np.log1p(x[big].clip(lower=0))
        return x
    sc = MinMaxScaler().fit(lg(df_tr))
    oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    oh.fit(df_tr[kcat])
    def f(d):
        a = np.clip(sc.transform(lg(d)), 0, 1)
        b = oh.transform(d[kcat])
        return np.hstack([a, b]).astype("float32")[..., None]
    return f(df_tr), f(df_va)

def build_cnn(width):
    m = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(width, 1)),
        tf.keras.layers.Conv1D(32, 3, activation="relu"),
        tf.keras.layers.Conv1D(64, 3, activation="relu"),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(kK, activation="softmax"),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy")
    return m

In [ ]:
need = ["ktr", "kfeats", "kcat", "kytr", "kK", "kn", "kg", "kfold", "A", "build_cnn", "cnn_prep"]
print({v: (v in globals()) for v in need})

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
kg = ktr.groupby(kfeats, dropna=False).ngroup().values
kfold = np.zeros(len(ktr), int)
sgk = StratifiedGroupKFold(10, shuffle=True, random_state=0)
for i, (_, v) in enumerate(sgk.split(ktr[kfeats], kytr, kg)):
    kfold[v] = i
print(np.bincount(kfold))

In [ ]:
with open("/content/KDDTrain+.txt") as f:
    print(repr(f.readline()))
print(ktr.iloc[0, :6].tolist())
print(ktr["label"].value_counts(dropna=False).head(5))

In [ ]:
def read_kdd(path):
    with open(path) as f:
        first = f.readline()
    sep = "," if first.count(",") > first.count("\t") else "\t"
    print(path, "separator:", repr(sep))
    return pd.read_csv(path, names=cols, sep=sep)

ktr = read_kdd(TR)
kte = read_kdd(TE)
print(ktr.shape, kte.shape)

In [ ]:
print(kte["label"].value_counts().head(15))
print(kte["label"].nunique(), "distinct labels in test")
print(ktr["label"].nunique(), "distinct labels in train")

In [ ]:
kte["cls"] = kte["label"].map(FAM).fillna("UNKNOWN")
print("unmapped test rows:", (kte.cls == "UNKNOWN").sum())
print(kte.loc[kte.cls == "UNKNOWN", "label"].value_counts())

print("\ntest class counts:")
print(kte.cls.value_counts())

seen = set(ktr["label"])
unseen = ~kte["label"].isin(seen)
print("\ntest rows with attack types never seen in train:", int(unseen.sum()),
      "of", len(kte))
print(kte.loc[unseen, "cls"].value_counts())

h1 = pd.util.hash_pandas_object(ktr[kfeats], index=False)
h2 = pd.util.hash_pandas_object(kte[kfeats], index=False)
print("\ntest rows with features identical to a train row:",
      int(h2.isin(set(h1)).sum()))

In [ ]:
import numpy as np, pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold

TR = "/content/KDDTrain+.txt"
TE = "/content/KDDTest+.txt"

cols = [
 "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
 "wrong_fragment","urgent","hot","num_failed_logins","logged_in",
 "num_compromised","root_shell","su_attempted","num_root",
 "num_file_creations","num_shells","num_access_files","num_outbound_cmds",
 "is_host_login","is_guest_login","count","srv_count","serror_rate",
 "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
 "diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count",
 "dst_host_same_srv_rate","dst_host_diff_srv_rate",
 "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate",
 "dst_host_serror_rate","dst_host_srv_serror_rate","dst_host_rerror_rate",
 "dst_host_srv_rerror_rate","label","difficulty",
]

def read_kdd(path):
    with open(path) as f:
        first = f.readline()
    sep = "," if first.count(",") > first.count("\t") else "\t"
    return pd.read_csv(path, names=cols, sep=sep)

ktr = read_kdd(TR)
kte = read_kdd(TE)
print(ktr.shape, kte.shape)

FAM = {"normal": "Normal"}
for a in ["back","land","neptune","pod","smurf","teardrop","apache2",
          "udpstorm","processtable","worm","mailbomb"]:
    FAM[a] = "DoS"
for a in ["satan","ipsweep","nmap","portsweep","mscan","saint"]:
    FAM[a] = "Probe"
for a in ["guess_passwd","ftp_write","imap","phf","multihop","warezmaster",
          "warezclient","spy","xlock","xsnoop","snmpguess","snmpgetattack",
          "httptunnel","sendmail","named"]:
    FAM[a] = "R2L"
for a in ["buffer_overflow","loadmodule","rootkit","perl","sqlattack",
          "xterm","ps"]:
    FAM[a] = "U2R"

ktr["cls"] = ktr["label"].map(FAM).fillna("UNKNOWN")
kte["cls"] = kte["label"].map(FAM).fillna("UNKNOWN")
print("unmapped train / test:", (ktr.cls == "UNKNOWN").sum(),
      (kte.cls == "UNKNOWN").sum())

kfeats = [c for c in cols if c not in ("label", "difficulty", "cls")]
kcat = ["protocol_type", "service", "flag"]
kle = LabelEncoder().fit(ktr["cls"])
kytr = kle.transform(ktr["cls"])
kyte = kle.transform(kte["cls"])
kK = len(kle.classes_)
kn = int(kle.transform(["Normal"])[0])
print("classes:", list(kle.classes_), "| Normal index:", kn)

kg = ktr.groupby(kfeats, dropna=False).ngroup().values
print("unique feature rows:", len(np.unique(kg)))

kfold = np.zeros(len(ktr), int)
sgk = StratifiedGroupKFold(10, shuffle=True, random_state=0)
for i, (_, v) in enumerate(sgk.split(ktr[kfeats], kytr, kg)):
    kfold[v] = i
print("fold sizes:", np.bincount(kfold))

In [ ]:
import os, time
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler, OneHotEncoder
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

OUT = "/content/drive/MyDrive/phd_data"
os.makedirs(OUT, exist_ok=True)
dev = "cuda" if tf.config.list_physical_devices("GPU") else "cpu"
print("xgboost device:", dev)

num_cols = [c for c in kfeats if c not in kcat]

A = ktr[kfeats].copy()
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
A[kcat] = oe.fit_transform(ktr[kcat])

def cnn_prep(df_tr, df_va):
    big = [c for c in num_cols if df_tr[c].max() > 1000]
    def lg(d):
        x = d[num_cols].copy()
        x[big] = np.log1p(x[big].clip(lower=0))
        return x
    sc = MinMaxScaler().fit(lg(df_tr))
    oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    oh.fit(df_tr[kcat])
    def f(d):
        a = np.clip(sc.transform(lg(d)), 0, 1)
        b = oh.transform(d[kcat])
        return np.hstack([a, b]).astype("float32")[..., None]
    return f(df_tr), f(df_va)

def build_cnn(width):
    m = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(width, 1)),
        tf.keras.layers.Conv1D(32, 3, activation="relu"),
        tf.keras.layers.Conv1D(64, 3, activation="relu"),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(kK, activation="softmax"),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy")
    return m

In [ ]:
ox = np.zeros((len(ktr), kK))
orf = np.zeros((len(ktr), kK))
ocn = np.zeros((len(ktr), kK))
t0 = time.time()
for k in range(10):
    a, b = kfold != k, kfold == k
    xg = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.1,
                       tree_method="hist", device=dev, random_state=0)
    ox[b] = xg.fit(A[a], kytr[a]).predict_proba(A[b])
    rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=0)
    orf[b] = rf.fit(A[a], kytr[a]).predict_proba(A[b])
    tf.keras.utils.set_random_seed(0)
    Xa, Xb = cnn_prep(ktr[a], ktr[b])
    m = build_cnn(Xa.shape[1])
    m.fit(Xa, kytr[a], epochs=60, batch_size=512, verbose=0)
    ocn[b] = m.predict(Xb, batch_size=2048, verbose=0)
    np.save(f"{OUT}/nsl_oof_xgb_s0.npy", ox)
    np.save(f"{OUT}/nsl_oof_rf_s0.npy", orf)
    np.save(f"{OUT}/nsl_oof_cnn_s0.npy", ocn)
    np.save(f"{OUT}/nsl_fold_s0.npy", kfold)
    print("fold", k, "done", round(time.time() - t0), "s")

In [ ]:
def meta_cv(parts):
    X = np.hstack(parts)
    out = np.zeros((len(kytr), kK))
    for k in range(10):
        tr_i, va_i = kfold != k, kfold == k
        lr = LogisticRegression(max_iter=2000).fit(X[tr_i], kytr[tr_i])
        out[va_i] = lr.predict_proba(X[va_i])
    return out

def rep(p):
    pr = p.argmax(1)
    f1 = f1_score(kytr, pr, average="macro")
    fpr = ((kytr == kn) & (pr != kn)).sum() / (kytr == kn).sum()
    return f1, fpr, (pr == kytr).mean()

systems = {
    "XGBoost raw": ox,
    "RF raw": orf,
    "CNN raw": ocn,
    "LR on XGB only": meta_cv([ox]),
    "Stack RF+XGB": meta_cv([orf, ox]),
    "Stack RF+XGB+CNN": meta_cv([orf, ox, ocn]),
}
for name, p in systems.items():
    f1, fpr, acc = rep(p)
    print(f"{name:18s} F1 {f1:.4f} FPR {fpr:.4f} acc {acc:.4f}")

In [ ]:
from sklearn.metrics import f1_score

pred = {k: v.argmax(1) for k, v in systems.items()}
show = ["XGBoost raw", "LR on XGB only", "Stack RF+XGB", "Stack RF+XGB+CNN"]

print("per-class F1 (classes:", list(kle.classes_), ")")
for k in show:
    f = f1_score(kytr, pred[k], average=None)
    print(f"{k:18s}", np.round(f, 4))
print("class sizes in train:", np.bincount(kytr))

G = kg.max() + 1
def gc(pr):
    ok = pr == kytr
    tp = np.bincount(kg[ok] * kK + kytr[ok], minlength=G*kK).reshape(G, kK)
    pc = np.bincount(kg * kK + pr, minlength=G*kK).reshape(G, kK)
    tc = np.bincount(kg * kK + kytr, minlength=G*kK).reshape(G, kK)
    return tp, pc, tc

cnt = {k: gc(pred[k]) for k in show}

def mf1(w, tp, pc, tc):
    TP, PC, TC = w @ tp, w @ pc, w @ tc
    f1 = np.where(PC + TC > 0, 2 * TP / np.maximum(PC + TC, 1), 0)
    return f1[TC > 0].mean()

rng = np.random.default_rng(0)
B = 1000
res = {k: [] for k in show}
for b in range(B):
    w = np.bincount(rng.integers(0, G, G), minlength=G).astype(float)
    for k in show:
        res[k].append(mf1(w, *cnt[k]))
res = {k: np.array(v) for k, v in res.items()}

def diff(a, b_):
    d = res[a] - res[b_]
    lo, hi = np.percentile(d, [2.5, 97.5])
    tag = "excludes 0" if lo > 0 or hi < 0 else "includes 0"
    print(f"{a} - {b_}: {d.mean():+.4f} CI [{lo:+.4f}, {hi:+.4f}] {tag}")

print()
diff("Stack RF+XGB+CNN", "Stack RF+XGB")
diff("Stack RF+XGB+CNN", "XGBoost raw")
diff("Stack RF+XGB+CNN", "LR on XGB only")
diff("XGBoost raw", "LR on XGB only")

In [ ]:
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

# ---- fit on FULL train, predict official test ----
At = kte[kfeats].copy()
At[kcat] = oe.transform(kte[kcat])

xg = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.1,
                   tree_method="hist", device=dev, random_state=0)
p_x = xg.fit(A, kytr).predict_proba(At)
rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=0)
p_r = rf.fit(A, kytr).predict_proba(At)
print("trees done")

tf.keras.utils.set_random_seed(0)
Xa, Xb = cnn_prep(ktr, kte)
m = build_cnn(Xa.shape[1])
m.fit(Xa, kytr, epochs=60, batch_size=512, verbose=0)
p_c = m.predict(Xb, batch_size=2048, verbose=0)
print("cnn done")

np.save(f"{OUT}/nsl_test_p_x.npy", p_x)
np.save(f"{OUT}/nsl_test_p_r.npy", p_r)
np.save(f"{OUT}/nsl_test_p_c.npy", p_c)

def meta(oofs, tests):
    lr = LogisticRegression(max_iter=2000).fit(np.hstack(oofs), kytr)
    return lr.predict_proba(np.hstack(tests))

sys_t = {
    "XGBoost raw": p_x,
    "RF raw": p_r,
    "CNN raw": p_c,
    "LR on XGB only": meta([ox], [p_x]),
    "Stack RF+XGB": meta([orf, ox], [p_r, p_x]),
    "Stack RF+XGB+CNN": meta([orf, ox, ocn], [p_r, p_x, p_c]),
}
pr_t = {k: v.argmax(1) for k, v in sys_t.items()}

# ---- report ----
unseen = ~kte["label"].isin(set(ktr["label"])).values
attack = kyte != kn
rows = []
for k, p in pr_t.items():
    f1 = f1_score(kyte, p, average="macro")
    fpr = ((kyte == kn) & (p != kn)).sum() / (kyte == kn).sum()
    acc = (p == kyte).mean()
    det_seen = (p[attack & ~unseen] != kn).mean()
    det_unseen = (p[attack & unseen] != kn).mean()
    rows.append([k, f1, fpr, acc, det_seen, det_unseen])
out = pd.DataFrame(rows, columns=["system", "macroF1", "FPR", "acc",
                                  "detect_seen_attacks", "detect_unseen_attacks"])
print(out.round(4).to_string(index=False))
out.to_csv(f"{OUT}/nsl_final_test_results.csv", index=False)

print("\nper-class F1 on test", list(kle.classes_))
for k, p in pr_t.items():
    print(f"{k:18s}", np.round(f1_score(kyte, p, average=None), 4))
print("test class sizes:", np.bincount(kyte))

# ---- McNemar + Holm on correctness ----
pairs = [("Stack RF+XGB+CNN", "Stack RF+XGB"),
         ("Stack RF+XGB+CNN", "XGBoost raw"),
         ("Stack RF+XGB", "XGBoost raw"),
         ("Stack RF+XGB", "LR on XGB only"),
         ("LR on XGB only", "XGBoost raw")]
pv, info = [], []
for a_, b_ in pairs:
    ca, cb = pr_t[a_] == kyte, pr_t[b_] == kyte
    t = [[(ca & cb).sum(), (ca & ~cb).sum()],
         [(~ca & cb).sum(), (~ca & ~cb).sum()]]
    pv.append(mcnemar(t, exact=False, correction=True).pvalue)
    info.append((a_, b_, t[0][1], t[1][0]))
rej, padj, _, _ = multipletests(pv, method="holm")
for (a_, b_, x, y), p, r in zip(info, padj, rej):
    print(f"{a_} vs {b_}: A-only right {x}, B-only right {y}, Holm p={p:.4g}",
          "significant" if r else "not significant")

In [ ]:
gte = pd.factorize(pd.util.hash_pandas_object(kte[kfeats], index=False))[0]
Gt = gte.max() + 1

def gct(pr):
    ok = pr == kyte
    tp = np.bincount(gte[ok] * kK + kyte[ok], minlength=Gt*kK).reshape(Gt, kK)
    pc = np.bincount(gte * kK + pr, minlength=Gt*kK).reshape(Gt, kK)
    tc = np.bincount(gte * kK + kyte, minlength=Gt*kK).reshape(Gt, kK)
    return tp, pc, tc

cnt_t = {k: gct(v) for k, v in pr_t.items()}

def mf1t(w, tp, pc, tc):
    TP, PC, TC = w @ tp, w @ pc, w @ tc
    f1 = np.where(PC + TC > 0, 2 * TP / np.maximum(PC + TC, 1), 0)
    return f1[TC > 0].mean()

rng = np.random.default_rng(0)
B = 2000
rt = {k: [] for k in cnt_t}
for b in range(B):
    w = np.bincount(rng.integers(0, Gt, Gt), minlength=Gt).astype(float)
    for k in cnt_t:
        rt[k].append(mf1t(w, *cnt_t[k]))
rt = {k: np.array(v) for k, v in rt.items()}

print("macro-F1 on official test, 95% CI")
for k, v in rt.items():
    lo, hi = np.percentile(v, [2.5, 97.5])
    print(f"{k:18s} {v.mean():.4f} [{lo:.4f}, {hi:.4f}]")

def dt(a_, b_):
    d = rt[a_] - rt[b_]
    lo, hi = np.percentile(d, [2.5, 97.5])
    tag = "excludes 0" if lo > 0 or hi < 0 else "includes 0"
    print(f"{a_} - {b_}: {d.mean():+.4f} CI [{lo:+.4f}, {hi:+.4f}] {tag}")

print()
dt("Stack RF+XGB+CNN", "XGBoost raw")
dt("Stack RF+XGB+CNN", "Stack RF+XGB")
dt("XGBoost raw", "LR on XGB only")
dt("XGBoost raw", "RF raw")

In [ ]:
Xa, Xb = cnn_prep(ktr, kte)

def score(pr):
    f1 = f1_score(kyte, pr, average="macro")
    fpr = ((kyte == kn) & (pr != kn)).sum() / (kyte == kn).sum()
    return f1, fpr

runs = []
for name, p in pr_t.items():
    runs.append((0, name) + score(p))

for seed in [1, 2]:
    t0 = time.time()
    xg = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.1,
                       tree_method="hist", device=dev, random_state=seed)
    p_x = xg.fit(A, kytr).predict_proba(At)
    rf = RandomForestClassifier(n_estimators=300, n_jobs=-1,
                                random_state=seed)
    p_r = rf.fit(A, kytr).predict_proba(At)
    tf.keras.utils.set_random_seed(seed)
    m = build_cnn(Xa.shape[1])
    m.fit(Xa, kytr, epochs=60, batch_size=512, verbose=0)
    p_c = m.predict(Xb, batch_size=2048, verbose=0)
    s = {
        "XGBoost raw": p_x,
        "RF raw": p_r,
        "CNN raw": p_c,
        "LR on XGB only": meta([ox], [p_x]),
        "Stack RF+XGB": meta([orf, ox], [p_r, p_x]),
        "Stack RF+XGB+CNN": meta([orf, ox, ocn], [p_r, p_x, p_c]),
    }
    for name, p in s.items():
        runs.append((seed, name) + score(p.argmax(1)))
    np.save(f"{OUT}/nsl_test_p_x_s{seed}.npy", p_x)
    np.save(f"{OUT}/nsl_test_p_r_s{seed}.npy", p_r)
    np.save(f"{OUT}/nsl_test_p_c_s{seed}.npy", p_c)
    print("seed", seed, "done", round(time.time() - t0), "s")

df = pd.DataFrame(runs, columns=["seed", "system", "macroF1", "FPR"])
print(df.pivot(index="system", columns="seed", values="macroF1").round(4))
print(df.groupby("system")[["macroF1", "FPR"]].agg(["mean", "std"]).round(4))

In [ ]:
import os
print(sorted(os.listdir("/content/drive/MyDrive/phd_data")))

In [ ]:
import os
print(sorted(os.listdir("/content/drive/MyDrive/phd_data")))

In [ ]:
F_X = find("oof_xgb*.npy")
F_R = find("oof_rf*.npy")
F_C = find("oof_cnn*.npy")

In [ ]:
F_X = D + "/oof_xgb.npy"
F_R = D + "/oof_rf.npy"
F_C = D + "/oof_cnn.npy"

In [ ]:
a = np.load(D + "/oof_xgb.npy"); b = np.load(D + "/oof_xgb (1).npy")
c = np.load(D + "/oof_rf.npy");  d = np.load(D + "/oof_rf (1).npy")
e = np.load(D + "/oof_cnn.npy"); f = np.load(D + "/oof_cnn (1).npy")
print(np.array_equal(a, b), np.array_equal(c, d), np.array_equal(e, f))

In [ ]:
import numpy as np, pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score

D = "/content/drive/MyDrive/phd_data"
lab = pd.read_csv(D + "/UNSW_NB15_training-set (1).csv")["attack_cat"]
ule = LabelEncoder().fit(lab)
uytr = ule.transform(lab)
un = int(ule.transform(["Normal"])[0])
print("rows", len(uytr), "classes", len(ule.classes_), "normal", un)

ra = np.load(D + "/oof_rf.npy")
rb = np.load(D + "/oof_rf (1).npy")

def cvscore(p):
    pr = p.argmax(1)
    f1 = f1_score(uytr, pr, average="macro")
    m = uytr == un
    fpr = (m & (pr != un)).sum() / m.sum()
    return round(f1, 4), round(fpr, 4)

print("A", cvscore(ra))
print("B", cvscore(rb))
print("diff", float(np.abs(ra - rb).max()))

fa = np.load(D + "/fold_id.npy")
fb = np.load(D + "/fold_id (1).npy")
print("folds equal:", np.array_equal(fa, fb))

In [ ]:
import os, time
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

D = "/content/drive/MyDrive/phd_data"
U_TR = D + "/UNSW_NB15_training-set (1).csv"
U_TE = D + "/UNSW_NB15_testing-set (1).csv"

utr = pd.read_csv(U_TR)
ute = pd.read_csv(U_TE)
ufeats = [c for c in utr.columns if c not in ("id", "attack_cat", "label")]
ucat = ["proto", "service", "state"]
unum = [c for c in ufeats if c not in ucat]
ule = LabelEncoder().fit(utr["attack_cat"])
uytr = ule.transform(utr["attack_cat"])
uyte = ule.transform(ute["attack_cat"])
uK = len(ule.classes_)
un = int(ule.transform(["Normal"])[0])
uox = np.load(D + "/oof_xgb.npy")
uorf = np.load(D + "/oof_rf.npy")
uocn = np.load(D + "/oof_cnn.npy")
print(len(utr), len(ute), uK, un, uox.shape, uorf.shape, uocn.shape)

uoe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
UA = utr[ufeats].copy()
UA[ucat] = uoe.fit_transform(utr[ucat])
UAt = ute[ufeats].copy()
UAt[ucat] = uoe.transform(ute[ucat])

ubig = [c for c in unum if utr[c].max() > 1000]

def ulog(d):
    x = d[unum].copy()
    x[ubig] = np.log1p(x[ubig].clip(lower=0))
    return x

usc = MinMaxScaler().fit(ulog(utr))
uoh = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
uoh.fit(utr[ucat])

def uprep(d):
    a = np.clip(usc.transform(ulog(d)), 0, 1)
    b = uoh.transform(d[ucat])
    return np.hstack([a, b]).astype("float32")[..., None]

UC = uprep(utr)
UCt = uprep(ute)

def ubuild(width):
    m = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(width, 1)),
        tf.keras.layers.Conv1D(32, 3, activation="relu"),
        tf.keras.layers.Conv1D(64, 3, activation="relu"),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(uK, activation="softmax"),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy")
    return m

def umeta(oofs, tests):
    lr = LogisticRegression(max_iter=2000).fit(np.hstack(oofs), uytr)
    return lr.predict_proba(np.hstack(tests))

print("load cell done")

In [ ]:
def uscore(pr):
    f1 = f1_score(uyte, pr, average="macro", labels=np.unique(uyte))
    m = uyte == un
    fpr = (m & (pr != un)).sum() / m.sum()
    return f1, fpr

uruns = []
for seed in [0, 1, 2]:
    t0 = time.time()
    xg = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.1,
                       tree_method="hist", n_jobs=-1, random_state=seed)
    p_x = xg.fit(UA, uytr).predict_proba(UAt)
    rf = RandomForestClassifier(n_estimators=300, n_jobs=-1,
                                random_state=seed)
    p_r = rf.fit(UA, uytr).predict_proba(UAt)
    tf.keras.utils.set_random_seed(seed)
    m = ubuild(UC.shape[1])
    m.fit(UC, uytr, epochs=60, batch_size=512, verbose=0)
    p_c = m.predict(UCt, batch_size=2048, verbose=0)
    s = {
        "XGBoost raw": p_x,
        "RF raw": p_r,
        "CNN raw": p_c,
        "LR on XGB only": umeta([uox], [p_x]),
        "Stack RF+XGB": umeta([uorf, uox], [p_r, p_x]),
        "Stack RF+XGB+CNN": umeta([uorf, uox, uocn], [p_r, p_x, p_c]),
    }
    for name, p in s.items():
        uruns.append((seed, name) + uscore(p.argmax(1)))
    np.save(D + f"/unsw_test_p_x_s{seed}.npy", p_x)
    np.save(D + f"/unsw_test_p_r_s{seed}.npy", p_r)
    np.save(D + f"/unsw_test_p_c_s{seed}.npy", p_c)
    print("seed", seed, "done", round(time.time() - t0), "s")

udf = pd.DataFrame(uruns, columns=["seed", "system", "macroF1", "FPR"])
print(udf.pivot(index="system", columns="seed", values="macroF1").round(4))
print(udf.groupby("system")[["macroF1", "FPR"]].agg(["mean", "std"]).round(4))

In [ ]:
import numpy as np
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
D="/content/drive/MyDrive/phd_data/"
N=int(ule.transform(["Normal"])[0])
pt=np.load(D+"unsw_test_p_x_s0.npy")
print(uox.shape, pt.shape, N)